In [ ]:
# STEP 0: Mount Google Drive
import os
try:
    from google.colab import drive
    drive.mount('/content/gdrive')
    project_path = '/content/gdrive/MyDrive/feral-cat-fcm'
except ImportError:
    project_path = os.path.abspath('feral-cat-fcm')  # running locally

# Define dataset and output paths
dataset_path = f'{project_path}/data/yolo_25-15-60'
crop_output_path = f'{project_path}/outputs/crops/manual'
class_names = ['back-leg', 'body', 'front-leg', 'tail']

In [ ]:
# STEP 1: Crop the manual labels of each cat's test set
from pathlib import Path
from PIL import Image

for cat_path in sorted(d for d in Path(dataset_path, 'test').iterdir() if d.is_dir()):
    count = 0
    for img_path in sorted((cat_path / 'images').iterdir()):
        label_path = cat_path / 'labels' / f'{img_path.stem}.txt'
        if img_path.suffix.lower() not in ('.jpg', '.jpeg', '.png') or not label_path.exists():
            continue
        img = Image.open(img_path).convert('RGB')
        w, h = img.size
        for i, line in enumerate(label_path.read_text().splitlines()):
            if not line.strip():
                continue
            cls_id, x_c, y_c, bw, bh = map(float, line.split())
            cls_id = int(cls_id)
            x1, y1 = int((x_c - bw / 2) * w), int((y_c - bh / 2) * h)
            x2, y2 = int((x_c + bw / 2) * w), int((y_c + bh / 2) * h)
            save_path = Path(crop_output_path, class_names[cls_id], cat_path.name)
            save_path.mkdir(parents=True, exist_ok=True)
            img.crop((x1, y1, x2, y2)).save(save_path / f'{img_path.stem}_crop_{i}_cls{cls_id}.jpg')
            count += 1
    print(f'{cat_path.name}: {count} crops')

In [ ]:
# STEP 2: Automatic vs manual crops per cat (auto / manual)
auto_path = f'{project_path}/outputs/crops/auto'


def count_crops(path):
    return {(part, cat.name): len(list(cat.iterdir()))
            for part in class_names if Path(path, part).is_dir()
            for cat in Path(path, part).iterdir() if cat.is_dir()}


auto, manual = count_crops(auto_path), count_crops(crop_output_path)
cats = sorted({cat for _, cat in auto} | {cat for _, cat in manual})
print('cat'.ljust(28) + ''.join(part.rjust(12) for part in class_names))
for cat in cats:
    print(cat.ljust(28) + ''.join(f'{auto.get((p, cat), 0)} / {manual.get((p, cat), 0)}'.rjust(12) for p in class_names))